# 02 — Build Bronze

Build five source-shaped Delta tables from the accepted landing artifacts: EMVS housing, population, district indicators, district boundaries, and the derived combined-grid snapshot.

Bronze preserves source values and observed quality conditions while adding row-level lineage. Business typing, filtering, identifier normalization, geometry repair, and analytical transformations remain Silver responsibilities.

## 1. Runtime and shared contracts

The landing volume remains the immutable byte-level source. Bronze makes its records queryable and adds source ID, source kind, collection method, landed path, SHA-256, run ID, and ingestion time. Reusable conversion and table operations live in `src/urban_workflow/bronze`.

All five fixed snapshots use overwrite semantics so an interactive rerun replaces the prior table instead of appending duplicate records. The development namespace is `workspace.urban_workflow`; notebook widgets and Lakeflow job parameters can override these values.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in [cwd, *cwd.parents] if (candidate / "src" / "urban_workflow").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the bundle root containing src/urban_workflow.")

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from urban_workflow.bronze.geospatial_operations import (
    read_geopackage_as_wkb,
    read_topojson_as_wkb,
)
from urban_workflow.bronze.table_operations import (
    add_bronze_metadata,
    apply_explicit_column_mapping,
    validate_bronze_table,
    write_snapshot_table,
)
from urban_workflow.ingestion.collect_local_source import collect_local_source
from urban_workflow.ingestion.collect_remote_source import validate_manually_uploaded_remote_source
from urban_workflow.ingestion.schemas import SourceManifest, resolve_manifest_paths

print(f"Bundle root: {repo_root}")

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "urban_workflow")
dbutils.widgets.text("raw_volume", "raw_sources")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
RAW_VOLUME = dbutils.widgets.get("raw_volume").strip()
if not all((CATALOG, SCHEMA, RAW_VOLUME)):
    raise ValueError("catalog, schema, and raw_volume must be non-empty.")
BRONZE_EMVS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_emvs_housing"
BRONZE_POPULATION_TABLE = f"{CATALOG}.{SCHEMA}.bronze_population_district_barrio"
BRONZE_INDICATOR_TABLE = f"{CATALOG}.{SCHEMA}.bronze_district_indicator_panel"
BRONZE_BOUNDARIES_TABLE = f"{CATALOG}.{SCHEMA}.bronze_district_boundaries"
BRONZE_GRID_TABLE = f"{CATALOG}.{SCHEMA}.bronze_combined_grid_snapshot"
RAW_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{RAW_VOLUME}"

bronze_started_at = datetime.now(timezone.utc)
bronze_run_id = (
    f"bronze_{bronze_started_at.strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Bronze run: {bronze_run_id}")
print("Target tables:")
for table_name in [
    BRONZE_EMVS_TABLE,
    BRONZE_POPULATION_TABLE,
    BRONZE_INDICATOR_TABLE,
    BRONZE_BOUNDARIES_TABLE,
    BRONZE_GRID_TABLE,
]:
    print(f"- {table_name}")

### Inputs and independent validation

The manifest and each source are revalidated so this notebook does not depend on hidden state from notebook 01. Independent inspection counts and SHA-256 provenance are reconciled with Spark and the persisted Delta tables.

In [ ]:
manifest_path = repo_root / "config" / "source_catalog.json"
with manifest_path.open("r", encoding="utf-8") as file_handle:
    manifest = SourceManifest.model_validate(json.load(file_handle))
manifest = resolve_manifest_paths(manifest, RAW_ROOT)

emvs_source = next(
    source for source in manifest.sources
    if source.source_id == "madrid_emvs_housing"
)
source_result = collect_local_source(
    specification=emvs_source,
    run_id=bronze_run_id,
    landing_root=Path(RAW_ROOT),
)

print(source_result.event.model_dump_json(indent=2))
if source_result.event.status != "succeeded":
    raise RuntimeError(source_result.event.error_message)
if source_result.provenance is None or source_result.event.record_count is None:
    raise RuntimeError("Successful source validation returned incomplete provenance.")

## 2. EMVS Bronze table

The EMVS CSV contains a preamble, a positional six-column schema, a `TOTAL` row, and two unnamed trailing positions. Bronze applies only the Delta-safe names required for those positions and keeps all 22 source values as strings.

The independent inspection, Spark parse, and persisted table counts must agree. The write is accepted only if the summary row, positional columns, source types, and lineage metadata survive persistence.

In [ ]:
source_header_mapping = [
    (1, "DISTRITOS", "DISTRITOS"),
    (2, "TOTAL", "TOTAL"),
    (3, "REGLAMENTO ADJUDICACION", "REGLAMENTO_ADJUDICACION"),
    (4, "RESTO PROGRAMAS", "RESTO_PROGRAMAS"),
    (5, "<empty>", "_UNNAMED_5"),
    (6, "<empty>", "_UNNAMED_6"),
]
source_column_names = [mapping[2] for mapping in source_header_mapping]
source_schema = StructType(
    [StructField(column_name, StringType(), nullable=True) for column_name in source_column_names]
)

display(
    spark.createDataFrame(
        source_header_mapping,
        ["source_position", "original_header", "bronze_column"],
    )
)

In [ ]:
if emvs_source.path is None:
    raise ValueError("EMVS manifest entry does not contain a source path.")

source_frame = (
    spark.read
    .format("csv")
    .schema(source_schema)
    .option("delimiter", ";")
    .option("encoding", "UTF-8")
    .option("skipRows", 2)
    .option("header", False)
    .option("mode", "FAILFAST")
    .load(emvs_source.path)
)

source_frame.printSchema()
display(source_frame)

In [ ]:
bronze_frame = add_bronze_metadata(
    source_frame,
    source_id=emvs_source.source_id,
    source_kind=emvs_source.source_kind,
    collection_method=source_result.event.collection_method,
    source_path=emvs_source.path,
    source_sha256=source_result.provenance.sha256,
    bronze_run_id=bronze_run_id,
    ingested_at_utc=bronze_started_at,
)

bronze_frame.printSchema()
display(bronze_frame)

In [ ]:
expected_source_count = source_result.event.record_count
spark_source_count = source_frame.count()
if spark_source_count != expected_source_count:
    raise ValueError(
        f"Source count mismatch: Python inspection found {expected_source_count}, "
        f"but Spark parsed {spark_source_count}."
    )

source_types = dict(source_frame.dtypes)
non_string_columns = [
    column_name for column_name in source_column_names
    if source_types.get(column_name) != "string"
]
if non_string_columns:
    raise TypeError(f"Bronze source columns must remain strings: {non_string_columns}")

print(f"Pre-write validation passed for {spark_source_count} source records.")

In [ ]:
write_snapshot_table(bronze_frame, BRONZE_EMVS_TABLE)

spark.sql(
    f"COMMENT ON TABLE {BRONZE_EMVS_TABLE} IS "
    "'Source-shaped EMVS housing records with Bronze ingestion metadata.'"
)
print(f"Wrote {BRONZE_EMVS_TABLE}.")

In [ ]:
persisted_frame = validate_bronze_table(
    spark,
    table_name=BRONZE_EMVS_TABLE,
    expected_record_count=expected_source_count,
    source_column_names=source_column_names,
)
persisted_count = persisted_frame.count()
total_row_count = persisted_frame.filter(F.col("DISTRITOS") == "TOTAL").count()
if total_row_count != 1:
    raise ValueError(f"Expected one preserved TOTAL row, found {total_row_count}.")

for unnamed_column in ["_UNNAMED_5", "_UNNAMED_6"]:
    if unnamed_column not in persisted_frame.columns:
        raise ValueError(f"Missing preserved source position: {unnamed_column}")

print(
    f"Bronze validation passed: source={expected_source_count}, "
    f"spark={spark_source_count}, delta={persisted_count}."
)
display(persisted_frame.orderBy(F.col("DISTRITOS").asc_nulls_last()))

## 3. CKAN Bronze tables

Each CKAN snapshot is a multiline JSON wrapper. Spark explodes `result.records` into table rows and preserves source-encoded JSON types. Envelope `success`, `offset`, `total`, and `limit` checks prevent incomplete pages from becoming Bronze tables.

Source-specific `column_mapping` declarations handle only technically unsafe names, including `Periodo panel` → `periodo_panel`. Python inspection, exploded Spark, and persisted Delta counts must agree.

In [ ]:
ckan_table_definitions = [
    {
        "source_id": "madrid_population_district_barrio_api",
        "table_name": BRONZE_POPULATION_TABLE,
        "column_mapping": {},
        "comment": "Source-shaped Madrid population CKAN records with Bronze ingestion metadata.",
    },
    {
        "source_id": "madrid_district_indicator_panel_api",
        "table_name": BRONZE_INDICATOR_TABLE,
        "column_mapping": {"Periodo panel": "periodo_panel"},
        "comment": "Source-shaped Madrid district indicator CKAN records with Bronze ingestion metadata.",
    },
]

In [ ]:
ckan_validation_overview = []
ckan_persisted_frames = {}

for table_definition in ckan_table_definitions:
    source_specification = next(
        source for source in manifest.sources
        if source.source_id == table_definition["source_id"]
    )
    validation_result = validate_manually_uploaded_remote_source(
        specification=source_specification,
        run_id=bronze_run_id,
        landing_root=Path(RAW_ROOT),
    )
    print(validation_result.event.model_dump_json(indent=2))
    if validation_result.event.status != "succeeded":
        raise RuntimeError(validation_result.event.error_message)
    if (
        validation_result.provenance is None
        or validation_result.ckan_page is None
        or validation_result.event.record_count is None
        or source_specification.manual_upload_path is None
    ):
        raise RuntimeError(
            f"Complete CKAN validation metadata is missing for {source_specification.source_id}."
        )

    payload_frame = (
        spark.read
        .option("multiLine", True)
        .option("mode", "FAILFAST")
        .json(source_specification.manual_upload_path)
    )
    envelope = (
        payload_frame
        .select(
            F.col("success").alias("success"),
            F.col("result.total").cast("long").alias("total"),
            F.col("result.limit").cast("long").alias("limit"),
            F.col("result.offset").cast("long").alias("offset"),
        )
        .first()
    )
    if envelope is None or envelope["success"] is not True:
        raise ValueError(f"CKAN envelope is not successful for {source_specification.source_id}.")

    expected_record_count = validation_result.event.record_count
    envelope_counts = {
        "total": envelope["total"],
        "limit": envelope["limit"],
        "offset": envelope["offset"],
    }
    if envelope_counts != {
        "total": expected_record_count,
        "limit": expected_record_count,
        "offset": 0,
    }:
        raise ValueError(
            f"Unexpected CKAN envelope for {source_specification.source_id}: "
            f"{envelope_counts}, expected total=limit={expected_record_count}, offset=0."
        )

    records_frame = (
        payload_frame
        .select(F.explode(F.col("result.records")).alias("record"))
        .select("record.*")
    )
    original_source_column_names = records_frame.columns
    if not original_source_column_names:
        raise ValueError(f"CKAN source contains no record fields: {source_specification.source_id}")

    records_frame = apply_explicit_column_mapping(
        records_frame,
        table_definition["column_mapping"],
    )
    source_column_names = records_frame.columns

    spark_record_count = records_frame.count()
    if spark_record_count != expected_record_count:
        raise ValueError(
            f"CKAN record mismatch for {source_specification.source_id}: Python found "
            f"{expected_record_count}, Spark exploded {spark_record_count}."
        )

    bronze_ckan_frame = add_bronze_metadata(
        records_frame,
        source_id=source_specification.source_id,
        source_kind=source_specification.source_kind,
        collection_method=validation_result.event.collection_method,
        source_path=source_specification.manual_upload_path,
        source_sha256=validation_result.provenance.sha256,
        bronze_run_id=bronze_run_id,
        ingested_at_utc=bronze_started_at,
    )
    write_snapshot_table(bronze_ckan_frame, table_definition["table_name"])
    table_comment = table_definition["comment"].replace("'", "''")
    spark.sql(
        f"COMMENT ON TABLE {table_definition['table_name']} IS '{table_comment}'"
    )

    persisted_ckan_frame = validate_bronze_table(
        spark,
        table_name=table_definition["table_name"],
        expected_record_count=expected_record_count,
        source_column_names=source_column_names,
    )
    persisted_record_count = persisted_ckan_frame.count()
    ckan_persisted_frames[source_specification.source_id] = persisted_ckan_frame
    ckan_validation_overview.append(
        {
            "source_id": source_specification.source_id,
            "table_name": table_definition["table_name"],
            "source_field_count": len(source_column_names),
            "renamed_source_fields": json.dumps(table_definition["column_mapping"]),
            "python_record_count": expected_record_count,
            "spark_record_count": spark_record_count,
            "delta_record_count": persisted_record_count,
            "source_sha256": validation_result.provenance.sha256,
        }
    )

display(spark.createDataFrame(ckan_validation_overview))

In [ ]:
for source_id, persisted_ckan_frame in ckan_persisted_frames.items():
    print(f"{source_id}: {persisted_ckan_frame.count()} persisted records")
    persisted_ckan_frame.printSchema()
    display(persisted_ckan_frame.limit(20))

## 4. Geospatial Bronze tables

GeoPandas and GDAL decode the 21 district boundaries and 10,100 grid cells on the driver. Geometry is stored as portable WKB in a Delta `BINARY` column, accompanied by mandatory geometry type, CRS, CRS evidence, and source-layer fields.

The TopoJSON adapter reconstructs polygons from shared arcs and applies the explicit EPSG:4326 contract after coordinate-bounds validation; the GeoPackage supplies EPSG:3035 in source metadata. Explicit schemas retain nullable fields such as `Area_m2`. Three self-intersecting boundary polygons are reported and preserved for Silver rather than silently repaired.

In [ ]:
grid_source = next(
    source for source in manifest.sources
    if source.source_id == "madrid_combined_grid_snapshot"
)
boundary_source = next(
    source for source in manifest.sources
    if source.source_id == "madrid_district_boundaries_remote"
)

grid_result = collect_local_source(
    specification=grid_source,
    run_id=bronze_run_id,
    landing_root=Path(RAW_ROOT),
)
print(grid_result.event.model_dump_json(indent=2))
if grid_result.event.status != "succeeded":
    raise RuntimeError(grid_result.event.error_message)

boundary_result = validate_manually_uploaded_remote_source(
    specification=boundary_source,
    run_id=bronze_run_id,
    landing_root=Path(RAW_ROOT),
)
print(boundary_result.event.model_dump_json(indent=2))
if boundary_result.event.status != "succeeded":
    raise RuntimeError(boundary_result.event.error_message)

if (
    grid_result.geopackage is None
    or grid_result.provenance is None
    or grid_result.event.record_count is None
    or grid_source.path is None
):
    raise RuntimeError("Complete GeoPackage validation metadata is required.")
if (
    boundary_result.json_file is None
    or boundary_result.provenance is None
    or boundary_result.event.record_count is None
    or boundary_source.manual_upload_path is None
):
    raise RuntimeError("Complete TopoJSON validation metadata is required.")

if grid_result.geopackage.crs != "EPSG:3035":
    raise ValueError(
        f"Unexpected grid CRS: {grid_result.geopackage.crs}; expected EPSG:3035."
    )

boundary_conversion = read_topojson_as_wkb(
    Path(boundary_source.manual_upload_path),
    topology_object="Distritos",
    expected_crs="EPSG:4326",
    allowed_geometry_types={"Polygon", "MultiPolygon"},
    required_attribute_columns={
        "COD_DIS",
        "COD_DIS_TX",
        "NOMBRE",
        "DISTRI_MAY",
        "DISTRI_MT",
        "Area_m2",
    },
)
grid_conversion = read_geopackage_as_wkb(
    Path(grid_source.path),
    layer=grid_result.geopackage.selected_layer,
    expected_crs="EPSG:3035",
    allowed_geometry_types={"Polygon", "MultiPolygon"},
    required_attribute_columns={
        "fid",
        "cell_id",
        "lu_2018_code",
        "lu_2018_class",
        "lu_2018_class_simplified",
        "height_mean",
        "height_max",
        "pt_stop_count",
        "pt_access_good",
        "rent_median_m2_2023",
        "emvs_units_total",
        "district_name",
    },
)

for source_id, conversion in [
    (boundary_source.source_id, boundary_conversion),
    (grid_source.source_id, grid_conversion),
]:
    print(
        f"{source_id}: {conversion.record_count} rows, "
        f"CRS={conversion.source_crs} ({conversion.crs_source}), "
        f"geometry_types={conversion.geometry_types}, "
        f"invalid_geometries={conversion.invalid_geometry_count}"
    )

In [ ]:
geometry_fields = [
    StructField("geometry_wkb", BinaryType(), nullable=False),
    StructField("geometry_type", StringType(), nullable=False),
    StructField("geometry_crs", StringType(), nullable=False),
    StructField("geometry_crs_source", StringType(), nullable=False),
    StructField("source_layer", StringType(), nullable=False),
]

boundary_schema = StructType(
    [
        StructField("COD_DIS", StringType(), nullable=True),
        StructField("COD_DIS_TX", StringType(), nullable=True),
        StructField("NOMBRE", StringType(), nullable=True),
        StructField("DISTRI_MAY", StringType(), nullable=True),
        StructField("DISTRI_MT", StringType(), nullable=True),
        StructField("Area_m2", DoubleType(), nullable=True),
        *geometry_fields,
    ]
)
grid_schema = StructType(
    [
        StructField("fid", LongType(), nullable=False),
        StructField("cell_id", LongType(), nullable=True),
        StructField("lu_2018_code", StringType(), nullable=True),
        StructField("lu_2018_class", StringType(), nullable=True),
        StructField("lu_2018_class_simplified", StringType(), nullable=True),
        StructField("height_mean", DoubleType(), nullable=True),
        StructField("height_max", DoubleType(), nullable=True),
        StructField("pt_stop_count", LongType(), nullable=True),
        StructField("pt_access_good", BooleanType(), nullable=True),
        StructField("rent_median_m2_2023", DoubleType(), nullable=True),
        StructField("emvs_units_total", DoubleType(), nullable=True),
        StructField("district_name", StringType(), nullable=True),
        *geometry_fields,
    ]
)

geospatial_table_definitions = [
    {
        "source_specification": boundary_source,
        "collection_result": boundary_result,
        "conversion": boundary_conversion,
        "schema": boundary_schema,
        "key_column": "COD_DIS_TX",
        "table_name": BRONZE_BOUNDARIES_TABLE,
        "comment": "Source-shaped Madrid district boundaries decoded from TopoJSON to WKB with Bronze metadata.",
    },
    {
        "source_specification": grid_source,
        "collection_result": grid_result,
        "conversion": grid_conversion,
        "schema": grid_schema,
        "key_column": "cell_id",
        "table_name": BRONZE_GRID_TABLE,
        "comment": "Preserved derived Madrid combined-grid GeoPackage snapshot encoded as WKB with Bronze metadata.",
    },
]

geospatial_validation_overview = []
geospatial_persisted_frames = {}

for table_definition in geospatial_table_definitions:
    source_specification = table_definition["source_specification"]
    collection_result = table_definition["collection_result"]
    conversion = table_definition["conversion"]
    expected_record_count = collection_result.event.record_count
    if expected_record_count is None:
        raise RuntimeError(f"Missing source count for {source_specification.source_id}.")
    if conversion.record_count != expected_record_count:
        raise ValueError(
            f"Geospatial conversion count mismatch for {source_specification.source_id}: "
            f"expected {expected_record_count}, converted {conversion.record_count}."
        )

    source_frame = spark.createDataFrame(
        conversion.records,
        schema=table_definition["schema"],
    )
    source_frame_count = source_frame.count()
    key_column = table_definition["key_column"]
    key_summary = source_frame.agg(
        F.count(F.col(key_column)).alias("non_null_count"),
        F.countDistinct(F.col(key_column)).alias("distinct_count"),
    ).first()
    if (
        source_frame_count != expected_record_count
        or key_summary["non_null_count"] != expected_record_count
        or key_summary["distinct_count"] != expected_record_count
    ):
        raise ValueError(
            f"{source_specification.source_id} requires {expected_record_count} "
            f"non-null unique {key_column} values; found {key_summary.asDict()}."
        )

    empty_wkb_count = source_frame.filter(
        F.col("geometry_wkb").isNull() | (F.length("geometry_wkb") == 0)
    ).count()
    if empty_wkb_count:
        raise ValueError(
            f"{source_specification.source_id} contains {empty_wkb_count} empty WKB values."
        )

    bronze_geospatial_frame = add_bronze_metadata(
        source_frame,
        source_id=source_specification.source_id,
        source_kind=source_specification.source_kind,
        collection_method=collection_result.event.collection_method,
        source_path=collection_result.event.landed_path,
        source_sha256=collection_result.provenance.sha256,
        bronze_run_id=bronze_run_id,
        ingested_at_utc=bronze_started_at,
    )
    write_snapshot_table(bronze_geospatial_frame, table_definition["table_name"])
    table_comment = table_definition["comment"].replace("'", "''")
    spark.sql(
        f"COMMENT ON TABLE {table_definition['table_name']} IS '{table_comment}'"
    )

    persisted_frame = validate_bronze_table(
        spark,
        table_name=table_definition["table_name"],
        expected_record_count=expected_record_count,
        source_column_names=source_frame.columns,
    )
    persisted_crs = {
        row["geometry_crs"]
        for row in persisted_frame.select("geometry_crs").distinct().collect()
    }
    if persisted_crs != {conversion.source_crs}:
        raise ValueError(
            f"Unexpected persisted CRS values for {source_specification.source_id}: "
            f"{persisted_crs}."
        )

    geospatial_persisted_frames[source_specification.source_id] = persisted_frame
    geospatial_validation_overview.append(
        {
            "source_id": source_specification.source_id,
            "source_kind": source_specification.source_kind,
            "table_name": table_definition["table_name"],
            "source_record_count": expected_record_count,
            "wkb_record_count": conversion.record_count,
            "delta_record_count": persisted_frame.count(),
            "geometry_types": ", ".join(conversion.geometry_types),
            "geometry_crs": conversion.source_crs,
            "geometry_crs_source": conversion.crs_source,
            "invalid_geometry_count": conversion.invalid_geometry_count,
            "source_sha256": collection_result.provenance.sha256,
        }
    )

display(spark.createDataFrame(geospatial_validation_overview))

In [ ]:
boundary_preview = geospatial_persisted_frames[boundary_source.source_id].select(
    "COD_DIS_TX",
    "NOMBRE",
    "geometry_type",
    "geometry_crs",
    "geometry_crs_source",
    F.substring(F.hex("geometry_wkb"), 1, 40).alias("wkb_hex_prefix"),
)
grid_preview = geospatial_persisted_frames[grid_source.source_id].select(
    "cell_id",
    "lu_2018_class_simplified",
    "district_name",
    "geometry_type",
    "geometry_crs",
    F.substring(F.hex("geometry_wkb"), 1, 40).alias("wkb_hex_prefix"),
)

display(boundary_preview.orderBy("COD_DIS_TX"))
display(grid_preview.orderBy("cell_id").limit(20))

dbutils.notebook.exit(
    "SUCCESS: Validated 5 Bronze tables: 22 EMVS, 1,071 population, "
    "116,930 indicator, 21 boundary, and 10,100 grid records."
)

## 5. Results and milestone status

Milestone 2 is complete. Databricks accepted all five Bronze tables with matching source and persisted counts, required lineage, deterministic snapshot writes, and the expected geospatial keys, WKB, and CRS contracts. The combined grid remains labelled `derived_snapshot`; notebook 03 performs Silver cleaning and normalization.